# BTL Hệ CSDL đa phương tiện - Nhóm 19: tìm kiếm ảnh chân dung đàn ông

Chạy lần lượt từ trên xuống. Mã nguồn nằm ở 4 tệp:

| Tệp | Nội dung |
|---|---|
| `common.py` | các phép xử lý tự code bằng numpy (ảnh xám, Laplace, căn chỉnh + nội suy, tô đa giác, RGB→HSV, LBP, Otsu) |
| `preprocess.py` | tiền xử lý B1 → B8 |
| `features.py` | vector đặc trưng 39 chiều |
| `search.py` | chuẩn hóa min-max, khoảng cách, top-5 |

Hình xuất ra thư mục `figs/` để chèn vào báo cáo.

In [ ]:
import os, csv
import numpy as np
import matplotlib.pyplot as plt

import common as C
import preprocess as P
import features as F
import search as S

os.makedirs("figs", exist_ok=True)
MA_MAU = "man_0097"        # ảnh dùng để minh họa các bước


def doc_log():
    with open(C.LOG_CSV, newline="") as f:
        return list(csv.DictReader(f))


def hien(a, anh, tieu_de=""):
    """Vẽ một ảnh đã căn chỉnh, chỉ hiện vùng quanh khuôn mặt (mặt luôn nằm ở cùng vị trí)."""
    a.imshow(anh); a.set_xlim(38, 218); a.set_ylim(236, 26); a.axis("off")
    a.set_title(tieu_de, fontsize=9)


def bang(tieu_de, hang):
    """In một bảng số liệu dạng văn bản."""
    print(" | ".join(tieu_de))
    for h in hang:
        print(" | ".join(f"{x:.4f}" if isinstance(x, float) else str(x) for x in h))

## 0. Đánh số ảnh (chỉ chạy 1 lần)
Chép ảnh thu thập được vào `data/raw256/`, đặt tên `man_0001.jpg`…; bảng đối chiếu tên gốc ở `data/name_map.csv`.

In [ ]:
# P.danh_so_anh("1000PictureFinal")      # bỏ dấu # nếu cần đánh số lại từ thư mục ảnh gốc
print("So anh tho:", len(os.listdir(C.RAW_DIR)))

## 1. Tiền xử lý (B1 → B8)
Mất khoảng 4 phút cho ~1000 ảnh. Kết quả lưu ở `data/prep/`.

In [ ]:
P.run()

### Bảng 1.3 - Số ảnh qua tiền xử lý, Bảng 1.4 - Thống kê trên các ảnh đạt

In [ ]:
log = doc_log()
dat = [r for r in log if r["dat"] == "1"]
print("Tong:", len(log), "| Dat:", len(dat))
ly_do = {}
for r in log:
    if r["dat"] != "1":
        ly_do[r["ly_do"]] = ly_do.get(r["ly_do"], 0) + 1
print("Ly do loai:", ly_do)
print("So anh dung cua so diem SVM < 0 (mat yeu):", sum(float(r["diem_mat"]) < 0 for r in dat))
print()
ten = {"goc_nghieng": "Goc nghieng truoc can chinh (do)", "do_net": "Do net (phuong sai Laplace)",
       "ti_le_khung": "Ti le khung mat dlib / anh (anh tho)", "lech_tam": "Lech tam khung mat (px, anh tho)",
       "he_so": "He so co gian s", "ti_le_mat": "Ti le mat na mat / anh (sau can chinh)",
       "lech_moc": "Do lech diem moc 2 lan do", "thieu": "Phan mat na thieu du lieu"}
hang = []
for k, t in ten.items():
    v = np.array([float(r[k]) for r in dat])
    hang.append([t, float(v.min()), float(v.max()), float(v.mean())])
bang(["Chi so", "Nho nhat", "Lon nhat", "Trung binh"], hang)

### Hình 1.2 - Một ảnh đi qua các bước

In [ ]:
rgb0 = P.read_rgb(os.path.join(C.RAW_DIR, MA_MAU + ".jpg"))                 # B1
gray0 = C.to_gray(rgb0)                                                     # B2
print("Do net:", round(C.laplacian_var(gray0), 1))                          # B3
so_mat, khung, pts0, diem = P.detect_landmarks(np.round(gray0).astype(np.uint8))   # B4, B5
trai, phai = C.eye_centers(pts0)                                            # B6
theta = np.arctan2(phai[1] - trai[1], phai[0] - trai[0])
s = C.D_MAT / C.dist(trai, phai)
m = (trai + phai) / 2
print(f"theta = {np.degrees(theta):.2f} do, s = {s:.3f}, trung diem hai mat = ({m[0]:.1f}; {m[1]:.1f})")
rgb1 = np.clip(np.round(C.align_image(rgb0, theta, s, m)), 0, 255).astype(np.uint8)
pts1 = C.align_points(pts0, theta, s, m)
rgb2 = rgb1.copy()                                                          # B7
rgb2[~P.face_mask(pts1)] = 255

fig, ax = plt.subplots(1, 5, figsize=(15, 3.4))
ax[0].imshow(rgb0); ax[0].set_title("B1. Ảnh gốc 256×256")
ax[1].imshow(gray0, cmap="gray", vmin=0, vmax=255); ax[1].set_title("B2. Ảnh xám")
ax[2].imshow(gray0, cmap="gray", vmin=0, vmax=255); ax[2].set_title("B4-B5. Khung mặt + 68 điểm")
l, t, r, b = khung
ax[2].plot([l, r, r, l, l], [t, t, b, b, t], "-", color="yellow", lw=1.5)
ax[2].plot(pts0[:, 0], pts0[:, 1], ".", color="lime", ms=4)
ax[3].imshow(rgb1); ax[3].set_title("B6. Căn chỉnh theo hai mắt")
ax[3].plot(pts1[:, 0], pts1[:, 1], ".", color="lime", ms=4)
ax[3].axhline(C.MAT_Y, color="yellow", lw=0.8, ls="--")
ax[4].imshow(rgb2); ax[4].set_title("B7. Lọc nền")
for a in ax:
    a.set_xlim(0, 255); a.set_ylim(255, 0); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig("figs/fig_steps.png", dpi=150); plt.show()

### Một số ảnh sau tiền xử lý (dữ liệu dùng để trích đặc trưng)

In [ ]:
ma_dat = sorted(F.read_landmarks())
chon = ma_dat[::len(ma_dat) // 12][:12]
fig, ax = plt.subplots(2, 6, figsize=(13, 4.6))
for a, ma in zip(ax.ravel(), chon):
    hien(a, F.read_prep(ma)[0], ma)
plt.tight_layout(); plt.savefig("figs/fig_dataset.png", dpi=130); plt.show()

## 2. Trích rút đặc trưng (39 chiều)

In [ ]:
F.run()

### Hình 2.1 - 68 điểm mốc; Hình 2.2 - các vùng lấy đặc trưng kết cấu và màu

In [ ]:
lm = F.read_landmarks()
rgb, gray = F.read_prep(MA_MAU)
pts = lm[MA_MAU]

fig, ax = plt.subplots(figsize=(7.5, 7.5))
ax.imshow(gray, cmap="gray", vmin=0, vmax=255, alpha=0.55)
nhom = [(0, 17, "tab:green", "viền hàm 0-16"), (17, 27, "tab:orange", "lông mày 17-26"),
        (27, 36, "tab:blue", "mũi 27-35"), (36, 48, "tab:red", "mắt 36-47"), (48, 68, "tab:purple", "miệng 48-67")]
for a, b, mau, ten_nhom in nhom:
    ax.plot(pts[a:b, 0], pts[a:b, 1], "o", color=mau, ms=4, label=ten_nhom)
    for i in range(a, b):
        ax.text(pts[i, 0] + 1.2, pts[i, 1] - 1.2, str(i), fontsize=7, color=mau)
x1, x2 = pts[:, 0].min() - 12, pts[:, 0].max() + 12
y1, y2 = pts[:, 1].min() - 22, pts[:, 1].max() + 10
ax.set_xlim(x1, x2); ax.set_ylim(y2, y1); ax.set_xticks([]); ax.set_yticks([])
ax.legend(loc="upper center", ncol=3, fontsize=8)
plt.tight_layout(); plt.savefig("figs/fig_landmarks.png", dpi=150); plt.show()

fig, ax = plt.subplots(1, 3, figsize=(12, 4.2))
to = np.zeros((C.SIZE, C.SIZE, 4))
to[F.chin_region(pts)] = (1, 0, 0, 0.45)
for v in F.brow_regions(pts):
    to[v] = (0, 0.4, 1, 0.5)
ax[0].imshow(gray, cmap="gray", vmin=0, vmax=255); ax[0].imshow(to)
ax[0].set_title("Kết cấu: cằm - miệng (đỏ), lông mày (xanh)")
ax[1].imshow(C.lbp(gray), cmap="gray"); ax[1].set_title("Ảnh mã LBP (0..255)")
to = np.zeros((C.SIZE, C.SIZE, 4)); to[F.skin_mask(pts)] = (0, 1, 0, 0.45)
ax[2].imshow(rgb); ax[2].imshow(to); ax[2].set_title("Màu: mặt nạ da (2 má + trán)")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig("figs/fig_regions.png", dpi=150); plt.show()

### Giá trị đặc trưng của ảnh mẫu và Bảng 2.5 - thống kê trên cả bộ dữ liệu

In [ ]:
ids, X = S.load_features()
v = X[ids.index(MA_MAU)]
print(MA_MAU, "- 7 dac trung dau:")
for ten_cot, gia_tri in zip(F.FEATURE_NAMES[:7], v[:7]):
    print(f"  {ten_cot:14s} = {gia_tri:.4f}")
print("  luoc do H x S (hang = o H 0..7, cot = o S 0..3):")
print(np.round(v[7:].reshape(F.SO_O_H, F.SO_O_S), 3))
print()
bang(["Dac trung", "Nho nhat", "Lon nhat", "Trung binh", "Do lech chuan"],
     [[n, float(X[:, j].min()), float(X[:, j].max()), float(X[:, j].mean()), float(X[:, j].std())]
      for j, n in enumerate(F.FEATURE_NAMES)])

### Thống kê sắc độ H của các điểm da (lý do chia ô H trong 0°-40°)

In [ ]:
dem = np.zeros(4)            # H < 40 | H >= 330 | 40 <= H < 330 | tổng
tb_luoc_do = X[:, 7:].mean(axis=0).reshape(F.SO_O_H, F.SO_O_S)
for ma in ids:
    rgb_i, _ = F.read_prep(ma)
    da = F.skin_mask(lm[ma]) & ~(rgb_i == 255).all(axis=2)
    h, s_, _ = C.rgb_to_hsv(rgb_i[da])
    dem += [(h < 40).sum(), (h >= 330).sum(), ((h >= 40) & (h < 330)).sum(), h.size]
print(f"So diem da trung binh moi anh: {dem[3] / len(ids):.0f}")
print(f"H < 40 do: {100 * dem[0] / dem[3]:.1f} % | H >= 330 do: {100 * dem[1] / dem[3]:.1f} % | 40 <= H < 330: {100 * dem[2] / dem[3]:.1f} %")
print("Ti le trung binh theo o H (0..7):", np.round(tb_luoc_do.sum(axis=1), 3))
print("Ti le trung binh theo o S (0..3):", np.round(tb_luoc_do.sum(axis=0), 3))

### Hình 1.3 - Ví dụ khuôn mặt giống và khác nhau về dáng mặt (đặc trưng `geo_khung_mat`)

In [ ]:
thu_tu = np.argsort(X[:, 0])
n = len(ids)
nhom = [("Mặt dài (tỉ lệ khung mặt thấp, quanh mức 10%)", thu_tu[n // 10 - 2:n // 10 + 3]),
        ("Trung bình (quanh trung vị)", thu_tu[n // 2 - 2:n // 2 + 3]),
        ("Mặt rộng (tỉ lệ khung mặt cao, quanh mức 90%)", thu_tu[9 * n // 10 - 2:9 * n // 10 + 3])]
fig, ax = plt.subplots(3, 5, figsize=(10, 7.6))
for hang, (ten_nhom, chi_so) in enumerate(nhom):
    for cot, i in enumerate(chi_so):
        hien(ax[hang, cot], F.read_prep(ids[i])[0], f"{ids[i]}   a = {X[i, 0]:.2f}")
    ax[hang, 2].set_title(ten_nhom + "\n" + ax[hang, 2].get_title(), fontsize=9)
plt.tight_layout(); plt.savefig("figs/fig_similar.png", dpi=130); plt.show()

## 3. Chuẩn hóa và tìm kiếm top-5
`S.fit_norm` tính min, max từng chiều trên toàn bộ CSDL và lưu `data/norm_params.json`.

In [ ]:
mn, mx = S.fit_norm(X)
Xn = S.normalize(X, mn, mx)
print("Sau chuan hoa: nho nhat", Xn.min(), "| lon nhat", Xn.max())


def hien_ket_qua(anh_truy_van, top, ten_tep):
    fig, ax = plt.subplots(1, 6, figsize=(13, 2.9))
    hien(ax[0], anh_truy_van, "Ảnh truy vấn")
    for a, r in zip(ax[1:], top):
        hien(a, F.read_prep(r["ma_anh"])[0], f"{r['ma_anh']}\nS = {r['S']:.4f}")
    plt.tight_layout(); plt.savefig(ten_tep, dpi=130); plt.show()


# Kiểm tra: lấy chính một ảnh trong CSDL làm truy vấn -> ảnh đó phải đứng đầu với S = 1
kq, v, top = S.search_image(os.path.join(C.RAW_DIR, MA_MAU + ".jpg"))
bang(["Anh", "S", "D", "D hinh hoc", "D ket cau", "D mau sac"],
     [[r["ma_anh"], r["S"], r["D"], r["D_hinh_hoc"], r["D_ket_cau"], r["D_mau_sac"]] for r in top])
hien_ket_qua(kq["rgb"], top, "figs/fig_top5.png")

### Truy vấn như một ảnh mới
Mỗi người chỉ có 1 ảnh nên để thử "ảnh mới", lấy một ảnh trong CSDL làm truy vấn và **không cho chính nó vào kết quả** (`bo_qua`).
In kèm kết quả trung gian: vector gốc, vector chuẩn hóa, khoảng cách từng bộ.
Với ảnh nằm ngoài CSDL: `kq, v, top = S.search_image("duong_dan.jpg")`.

In [ ]:
MA_TRUY_VAN = "man_0050"
v = X[ids.index(MA_TRUY_VAN)]
q = S.normalize(v, *S.load_norm())
top = S.search_vector(v, bo_qua=MA_TRUY_VAN)
print("7 dac trung dau (goc)      :", np.round(v[:7], 4))
print("7 dac trung dau (chuan hoa):", np.round(q[:7], 4))
bang(["Anh", "S", "D", "D hinh hoc", "D ket cau", "D mau sac"],
     [[r["ma_anh"], r["S"], r["D"], r["D_hinh_hoc"], r["D_ket_cau"], r["D_mau_sac"]] for r in top])
hien_ket_qua(F.read_prep(MA_TRUY_VAN)[0], top, "figs/fig_query.png")